# Polymarket Data Pull Notebook

This notebook pulls public Polymarket data using only the Python standard library.

It covers:
- market metadata from the Gamma API
- trade history from the Data API
- price history from the CLOB API
- an optional binary-market trajectory build, without `requests`, `numpy`, or `pandas`

Edit the `SLUG` cell and run the notebook from top to bottom.

In [1]:
import json
from datetime import datetime, timezone
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

In [2]:
GAMMA_API = "https://gamma-api.polymarket.com"
DATA_API = "https://data-api.polymarket.com"
CLOB_API = "https://clob.polymarket.com"


def fetch_json(base_url, path, params=None, timeout=20):
    query = urlencode(params or {}, doseq=True)
    url = f"{base_url}{path}"
    if query:
        url = f"{url}?{query}"
    request = Request(
        url,
        headers={
            "User-Agent": "python-stdlib-polymarket-notebook/1.0",
            "Accept": "application/json",
        },
    )
    with urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


def parse_json_list(value):
    if isinstance(value, list):
        return value
    if value in (None, ""):
        return []
    if isinstance(value, str):
        return json.loads(value)
    raise TypeError(f"cannot parse JSON list from {type(value)!r}")


def pretty_print(value):
    print(json.dumps(value, indent=2, sort_keys=True))


def iso_utc(timestamp):
    return datetime.fromtimestamp(int(timestamp), tz=timezone.utc).isoformat()


def clip_probability(value, epsilon=1e-6):
    value = float(value)
    if value < epsilon:
        return epsilon
    if value > 1.0 - epsilon:
        return 1.0 - epsilon
    return value

In [3]:
def normalize_market(market):
    outcomes = [str(item) for item in parse_json_list(market.get("outcomes"))]
    outcome_prices = [float(item) for item in parse_json_list(market.get("outcomePrices"))]
    token_ids = [str(item) for item in parse_json_list(market.get("clobTokenIds"))]
    return {
        "question": str(market["question"]),
        "slug": str(market["slug"]),
        "condition_id": str(market["conditionId"]),
        "outcomes": outcomes,
        "outcome_prices": outcome_prices,
        "token_ids": token_ids,
        "closed": bool(market.get("closed", False)),
        "active": bool(market.get("active", False)),
        "volume": float(market.get("volumeNum", market.get("volume", 0.0)) or 0.0),
        "raw": market,
    }


def get_market_by_slug(slug, timeout=20):
    markets = fetch_json(GAMMA_API, "/markets", {"slug": slug, "limit": 5}, timeout=timeout)
    if markets:
        return normalize_market(markets[0])

    events = fetch_json(GAMMA_API, "/events", {"slug": slug, "limit": 5}, timeout=timeout)
    if not events:
        raise ValueError(f"no Polymarket market found for slug {slug!r}")

    nested_markets = events[0].get("markets") or []
    if not nested_markets:
        raise ValueError(f"event {slug!r} did not contain any markets")

    return normalize_market(nested_markets[0])


def is_binary_market(market):
    return len(market["outcomes"]) == 2 and len(market["token_ids"]) == 2


def resolved_outcome_index(market, threshold=0.99):
    if not is_binary_market(market):
        raise ValueError("only binary markets are supported")
    for index, price in enumerate(market["outcome_prices"]):
        if float(price) >= threshold:
            return index
    raise ValueError("market does not appear resolved to a single outcome")

In [4]:
def get_trades(condition_id, page_size=500, max_pages=100, timeout=20):
    trades = []
    seen = set()

    for page in range(max_pages):
        offset = page * page_size
        try:
            batch = fetch_json(
                DATA_API,
                "/trades",
                {
                    "market": condition_id,
                    "limit": page_size,
                    "offset": offset,
                },
                timeout=timeout,
            )
        except HTTPError as exc:
            if exc.code == 400 and page > 0:
                break
            raise

        if not batch:
            break

        for trade in batch:
            key = (
                trade.get("transactionHash"),
                trade.get("asset"),
                trade.get("timestamp"),
                trade.get("price"),
                trade.get("size"),
            )
            if key not in seen:
                seen.add(key)
                trades.append(trade)

        if len(batch) < page_size:
            break

    trades.sort(key=lambda row: (int(row["timestamp"]), str(row.get("transactionHash", ""))))
    return trades


def get_price_history(token_id, fidelity_minutes=1, interval="max", timeout=20):
    payload = fetch_json(
        CLOB_API,
        "/prices-history",
        {
            "market": token_id,
            "interval": interval,
            "fidelity": fidelity_minutes,
        },
        timeout=timeout,
    )
    return list(payload.get("history", []))

In [5]:
def build_binary_trajectory(market, trades, bucket_minutes=1, resolve_threshold=0.99, price_history=None):
    if bucket_minutes <= 0:
        raise ValueError("bucket_minutes must be positive")
    if not is_binary_market(market):
        raise ValueError("only binary markets are supported")
    if not trades:
        raise ValueError("trades must not be empty")

    winner_index = resolved_outcome_index(market, threshold=resolve_threshold)
    bucket_seconds = int(bucket_minutes * 60)
    buckets = {}

    ordered_trades = sorted(
        trades,
        key=lambda row: (int(row["timestamp"]), str(row.get("transactionHash", ""))),
    )

    for trade in ordered_trades:
        timestamp = int(trade["timestamp"])
        price = float(trade["price"])
        size = float(trade["size"])
        outcome_index = int(trade["outcomeIndex"])
        implied_target_price = price if outcome_index == winner_index else 1.0 - price
        bucket = (timestamp // bucket_seconds) * bucket_seconds + bucket_seconds

        entry = buckets.setdefault(
            bucket,
            {"price": None, "volume": 0.0, "trade_count": 0},
        )
        entry["price"] = clip_probability(implied_target_price)
        entry["volume"] += size
        entry["trade_count"] += 1

    history_by_bucket = {}
    if price_history:
        for point in price_history:
            timestamp = int(point["t"])
            bucket = (timestamp // bucket_seconds) * bucket_seconds + bucket_seconds
            history_by_bucket[bucket] = clip_probability(point["p"])

    bucket_keys = list(range(min(buckets), max(buckets) + bucket_seconds, bucket_seconds))
    rows = []

    for bucket in bucket_keys:
        trade_row = buckets.get(bucket, {"price": None, "volume": 0.0, "trade_count": 0})
        price = trade_row["price"]
        if price is None and bucket in history_by_bucket:
            price = history_by_bucket[bucket]
        rows.append(
            {
                "bucket": bucket,
                "price": price,
                "volume": float(trade_row["volume"]),
                "trade_count": int(trade_row["trade_count"]),
            }
        )

    known_prices = [row["price"] for row in rows if row["price"] is not None]
    if not known_prices:
        raise ValueError("unable to construct price series from trades or price history")

    current_price = known_prices[0]
    for row in rows:
        if row["price"] is None:
            row["price"] = current_price
        else:
            current_price = row["price"]

    prices = [rows[0]["price"]] + [row["price"] for row in rows]
    volumes = [row["volume"] for row in rows]
    times = [rows[0]["bucket"] - bucket_seconds] + [row["bucket"] for row in rows]

    return {
        "winner_index": winner_index,
        "winner_label": market["outcomes"][winner_index],
        "prices": prices,
        "volumes": volumes,
        "times": times,
        "horizon": len(volumes),
        "rows": rows,
        "metadata": {
            "source": "polymarket",
            "question": market["question"],
            "slug": market["slug"],
            "condition_id": market["condition_id"],
            "outcomes": market["outcomes"],
            "trade_count": len(trades),
            "price_from_history": bool(price_history),
        },
    }

In [6]:
# Edit these values.
SLUG = "btc-updown-5m-1776134700"
TIMEOUT = 20
TRADE_LIMIT = 10
HISTORY_LIMIT = 10
BUCKET_MINUTES = 1

In [7]:
market = get_market_by_slug(SLUG, timeout=TIMEOUT)
pretty_print(
    {
        "question": market["question"],
        "slug": market["slug"],
        "condition_id": market["condition_id"],
        "outcomes": market["outcomes"],
        "outcome_prices": market["outcome_prices"],
        "token_ids": market["token_ids"],
        "closed": market["closed"],
        "active": market["active"],
        "volume": market["volume"],
    }
)

{
  "active": true,
  "closed": true,
  "condition_id": "0x0de9798c1f1e0fa1bb899242b01b220b53265019e08c408a1c244dec316d77d8",
  "outcome_prices": [
    0.0,
    1.0
  ],
  "outcomes": [
    "Up",
    "Down"
  ],
  "question": "Bitcoin Up or Down - April 13, 10:45PM-10:50PM ET",
  "slug": "btc-updown-5m-1776134700",
  "token_ids": [
    "111119808277703656652943730286671537824957919011094511521340797209212803960330",
    "103046332391802538265813540280315261611756645217502168365218052256656251752411"
  ],
  "volume": 244805.2903509991
}


In [8]:
trades = get_trades(market["condition_id"], timeout=TIMEOUT)
print(f"Fetched {len(trades)} trades for condition_id={market['condition_id']}")
pretty_print(trades[:TRADE_LIMIT])

Fetched 3500 trades for condition_id=0x0de9798c1f1e0fa1bb899242b01b220b53265019e08c408a1c244dec316d77d8
[
  {
    "asset": "111119808277703656652943730286671537824957919011094511521340797209212803960330",
    "bio": "",
    "conditionId": "0x0de9798c1f1e0fa1bb899242b01b220b53265019e08c408a1c244dec316d77d8",
    "eventSlug": "btc-updown-5m-1776134700",
    "icon": "https://polymarket-upload.s3.us-east-2.amazonaws.com/BTC+fullsize.png",
    "name": "0x3A847382ad6FfF9be1db4e073FD9b869f6884D4",
    "outcome": "Up",
    "outcomeIndex": 0,
    "price": 0.29,
    "profileImage": "",
    "profileImageOptimized": "",
    "proxyWallet": "0x3a847382ad6fff9be1db4e073fd9b869f6884d44",
    "pseudonym": "Twin-Driving-Shower",
    "side": "BUY",
    "size": 63.57496,
    "slug": "btc-updown-5m-1776134700",
    "timestamp": 1776134847,
    "title": "Bitcoin Up or Down - April 13, 10:45PM-10:50PM ET",
    "transactionHash": "0x0261fa019211304ad6186a0e1014f1f478ee1474f94e3c205f88e38f6dfc0e63"
  },
  {
  

In [ ]:
if market["token_ids"]:
    token_id = market["token_ids"][0]
    history = get_price_history(token_id, fidelity_minutes=BUCKET_MINUTES, timeout=TIMEOUT)
    print(f"Fetched {len(history)} price points for token_id={token_id}")
    pretty_print(history[:HISTORY_LIMIT])
else:
    history = []
    print("This market did not expose any token ids.")

In [ ]:
if is_binary_market(market):
    winner_index = resolved_outcome_index(market)
    winner_token_id = market["token_ids"][winner_index]
    winner_history = get_price_history(
        winner_token_id,
        fidelity_minutes=BUCKET_MINUTES,
        timeout=TIMEOUT,
    )
    trajectory = build_binary_trajectory(
        market,
        trades,
        bucket_minutes=BUCKET_MINUTES,
        price_history=winner_history,
    )
    pretty_print(
        {
            "winner_index": trajectory["winner_index"],
            "winner_label": trajectory["winner_label"],
            "horizon": trajectory["horizon"],
            "initial_price": trajectory["prices"][0],
            "final_price": trajectory["prices"][-1],
            "total_volume": sum(trajectory["volumes"]),
            "start_time": iso_utc(trajectory["times"][0]),
            "end_time": iso_utc(trajectory["times"][-1]),
        }
    )
    pretty_print(trajectory["rows"][:10])
else:
    print("Market is not binary, so the optional trajectory cell is skipped.")